# **Analítica Computacional para la Toma de Decisiones**

## Taller 10: Visualización

Daniel Benavides -202220428

<d.benavidess@uniandes.edu.co>

In [38]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import altair as alt

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

import warnings
warnings.filterwarnings('ignore')  

In [39]:
# Read data

df = pd.read_csv("people.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [40]:
# Categorical and numerical variables
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [41]:
categorical_vars = df.select_dtypes(include=['object']).columns
numerical_vars = df.select_dtypes(include=['int64', 'float64']).columns

In [42]:
categorical_vars

Index(['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender',
       'JobRole', 'MaritalStatus', 'Over18', 'OverTime'],
      dtype='object')

In [43]:
numerical_vars

Index(['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
       'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='object')

In [44]:
variables = ["OverTime", "WorkLifeBalance", "YearsSinceLastPromotion"]

In [45]:
# Altair pairplot
num_vars = [
    'Age',
    'MonthlyIncome',
    'TotalWorkingYears',
    'YearsAtCompany',
    'YearsSinceLastPromotion',
    'WorkLifeBalance',
    'JobSatisfaction'
]

color_scale = alt.Scale(
    domain=['No', 'Yes'],
    range=['steelblue', 'crimson']
)

pairplot = (
    alt.Chart(df)
    .mark_circle(opacity=0.4, size=18)
    .encode(
        x=alt.X(alt.repeat('column'), type='quantitative'),
        y=alt.Y(alt.repeat('row'),    type='quantitative'),
        color=alt.Color('Attrition:N', scale=color_scale,
                        legend=alt.Legend(title='Attrition')),
        tooltip=['Age', 'MonthlyIncome', 'Attrition']
    )
    .properties(width=120, height=120)
    .repeat(
        row=num_vars,
        column=num_vars
    )
)

pairplot

alt.RepeatChart(...)

In [46]:
base = (
    alt.Chart(df)
    .transform_aggregate(count_="count()", groupby=["Department", "Attrition"])
    .transform_stack(
        stack="count_",
        as_=["stack_count_Department1", "stack_count_Department2"],
        offset="normalize",
        sort=[alt.SortField("Department", "ascending")],
        groupby=[],
    )
    .transform_window(
        x="min(stack_count_Department1)",
        x2="max(stack_count_Department2)",
        rank_Attrition="dense_rank()",
        distinct_Attrition="distinct(Attrition)",
        groupby=["Department"],
        frame=[None, None],
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_window(
        rank_Department="dense_rank()",
        frame=[None, None],
        sort=[alt.SortField("Department", "ascending")],
    )
    .transform_stack(
        stack="count_",
        groupby=["Department"],
        as_=["y", "y2"],
        offset="normalize",
        sort=[alt.SortField("Attrition", "ascending")],
    )
    .transform_calculate(
        ny="datum.y + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        ny2="datum.y2 + (datum.rank_Attrition - 1) * datum.distinct_Attrition * 0.01 / 3",
        nx="datum.x + (datum.rank_Department - 1) * 0.01",
        nx2="datum.x2 + (datum.rank_Department - 1) * 0.01",
        xc="(datum.nx + datum.nx2) / 2",
        yc="(datum.ny + datum.ny2) / 2",
    )
)

rect = base.mark_rect().encode(
    x=alt.X("nx:Q").axis(None),
    x2="nx2",
    y=alt.Y("ny:Q").axis(None),
    y2="ny2",
    color=alt.Color("Department:N", legend=None),         
    opacity=alt.condition(
        alt.datum.Attrition == "Yes",
        alt.value(0.8), 
        alt.value(0.5),  
    ),
    tooltip=[
        alt.Tooltip("Department:N",  title="Departamento"),
        alt.Tooltip("Attrition:N",   title="Attrition"),
        alt.Tooltip("count_:Q",      title="Empleados"),
    ],
)

# Etiqueta: "Yes" / "No"
text_label = base.mark_text(
    baseline="middle", align="center",
    fontSize=12, fontWeight="bold", dy=-10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="Attrition:N",
    color=alt.value("white"),
)

# Número de empleados en cada bloque
text_count = base.mark_text(
    baseline="middle", align="center",
    fontSize=14, fontWeight="bold", dy=10
).encode(
    x=alt.X("xc:Q").axis(None),
    y=alt.Y("yc:Q"),
    text="count_:Q",
    color=alt.value("white"),
)

mosaic = (rect + text_label + text_count).properties(width=620, height=420)

dept_labels = (
    base.mark_text(baseline="middle", align="center", fontSize=15, fontWeight="bold")
    .encode(
        x=alt.X("min(xc):Q").axis(None),
        color=alt.Color("Department:N", legend=None),
        text="Department:N",
    )
    .properties(width=620)
)

(
    (dept_labels & mosaic)
    .resolve_scale(x="shared")
    .configure_view(stroke="")
    .configure_concat(spacing=2)
    .configure_axis(domain=False, ticks=False, labels=False, grid=False)
)

alt.VConcatChart(...)